# GwenLand glcuda Wave 119 - T4 kernel profile

Observation-only event, Nsight Compute, resource, and SASS evidence.


In [ ]:
import base64
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import statistics
import subprocess
import traceback
import urllib.request
import zipfile

BUILD = "wave119-in-process-stability-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "5de5be39c0190b9367da18f0f2001e7f40208c92"
SOURCE_REV = "6b28171e160f7e5dd98c4c82703a6360f8edddde"
PATCH_SHA256 = "caafd7eb303d07e184b3d938c4fca84840234dca22bb4aa2a513d8087240c573"
PATCH_GZIP_B64 = """H4sIADwApmoC/71a63LbuBX+76dA1BmXWkvU/Uavs+sk3ky66zi1nW5nvB4aIkEJNS8KQflSxzN9iD5hn6TnACAJSrSd3ZnWPywJBA4OzvU7B/R5EJB2e8EzQjuL0Fv7tMPuaLQKmejc0hvW601dHrurNPGYEK7I6JyHPLu3U0Hmv3fFTsxuScBDRqLEZ6TX7Y6Hwx0e++yOdL/xz7bHwaw77/amUxb02LA3Dmi/P5kyOuuP+7PudDLqz4JhnwY77XabdHx204nXYbizt7f3Bzj+8UfS7ra6ZK/X6g1m5Mcfd/Y6nVfkV1hGYB3hcVuvI/Dpr72MJzEpSJAlTWN4aMtlau1JrM4ftki6jmOWtsjbz+8OYT2PaHpPvCTO2F3WIjT2CQ3DxKOSKA0zlsY0YyRbMkUqZRnlMfPlVJ8FLE2Z306Z4P6ahmRFs6WwydskimC9wV8Q0oUgNGVErFerkDNf0ZvfI23iwa4sJXMWJDAlPx8cKs329YS1APrilmfeknBB2B1Q8cCKlixlcNidvbVgBIQNBByHxQvg0s1SyjPHeXgfHsmBFvkQA8sf4tU6e9wvloB+5CT88jaJA75oEfVLLSunImeO815+qmf7uDUIUGTk0+nJ8adz9/PHD+cO2RVZSg5I45hRsU5RgsC0z0CgEY+5yLhHxL3IWCTVGK0yOGLKArCbe5scweFAzGSZ3JIsuWagcpqiiEJQf8YWIKqIZim/I9E6zDhKQmkMuASxgQmAhiIWJel9i5yzWCQp6AS0pFQc8Dt4HtJ1DLJcsCRiWQq7Nvbzk/z18+Hp+dH5mQME+T8ZnGPULR7+enh6/PnTmfvp6NSFr8acYsrR3z8dvT0/eudqkZyf/Hz00aDWHw6l3IIYzgO6sLgvQGQX60H/sknarw01kYedPQJ/2yP4J4XjytXwz84S94Z5VrNVzojonQtBwJUzYVrPeAbSX7GUZqAfh3TtrvkoWbnXDhlujq1w4mxkjKZsxWjmrlgM7nIPG9jmFrZdMu444DAUFGY11YTHnb1HLQbwS5emkaUeKPMFiZhWqKkaotIjHuiU++CmDpknSahHk5R6IQzBRBiRUj1lAnb/PhgPW+RNcve9f4+Bwwd3SdMkdZwj/Hj9Ohew4sIWLHPnDEwFYsW1K33eDYLYzZ3eKvZv/rCvVoYsIwlo6iCnwVEIVqHrZjGTBzjRVi6glUReHTxhQeTrVzm9ULtNIXqC9TOriasu1KEvTROBiLVOYwJnsyC4gNu8ssqH+NdQi0jEBTz1lg6Eqejg4ZFUmMKB/JvzwyPGH+ZlzD+4eHi8bLSqJOFUhVDIA2kUPxoEVoZCDuahFMY2lm9JpOZ5IYLNZ/IwxmDTMEYIHonVbO4X5ocfJ9eW2pBBogzdSDRLs4yAQ8u0HfCtbzEdNIAIDICmEPUP9Mz4xnFwwGra4pqvrF7TMBeZn2AqTjA4jiEvWeYRkms3Sa0GZIMFGPdzyZQcn7w7+oVc0K/zy0bFNCUWUFvpDYyHKbthqcDn0hqYeGXhfLQ0H1JNgBI4g5BpNeaNprkQGECk4Sax9y2r9XRFo0IFk8GBmU1sFWOsaUVgIN7Fag0zzSDhOJAhl64ns5hlpjTTJwRjvqP4GPbNaPmcn+sFAQX7zdc85hwBJ2BdHET5gzESJtR3pWatXflRUQMYL3CP82BXeOp6SzjkrpKAGSHQ0UMWKx9/IjL8Lo/PMy7gHnCXjPgpDzKHgIvDBg+PW+6c778xXs/KN7peVZOe1mQJLBwH0lZhmd6WfL0X5atCgasCsSJQRuHdShgup8PU78qFxgGKgGOMBTwVte6pkFJOcsFiTLOAOeJEBdFG5Rhiuc785Da2SlcAxRkhFKDQhbS7FsnSdTW+40wXZwBQ34Am5jRpGDrJ7oLNtQhKoFVu0tLcFpwZIbJQV6dDTuaCpTcSbrWTOLyHFAfoiawS0LBkRiP1mU1+ZmxFEgDf4L/gRTARlt2wgpR5QNAMw8ABB0G8W0DgBBAbJBsJxMM8xkjZgBeHkIIKajyrVATXsAJSDcF4gCYAPIT0nscLSb/f7baFrEiM0gHkYxdOVwlnhiBVkMpzBZhLnVRRS7UChZIjzsJ4yyMvVCSftfW2oOKH3xqVJPhbw3l4bOFgvjmOOPbMHMxWenSMo9qQ5Xq5/PGbfbvc5akFhAoCUIp8h2UloEfSeXrRdlY2ZLLYdIGNQAYJ+onIEWN4w6xpekf+Xw5dbmc2H+cbU4oP6VqVgAApCyaDMeRJ8aEkUgAZzcSjGdnjm7yG3FgOSdOAQLRRLEPP+bKGek8uR2fOi5BNZ7dWieBI23DeJq6R/MpA63LwF4AZLF5HMvKAtjZiwTeZ8fPBIScDq9Qxfxfm29+0t1rHMJ1jqj1W+UYpY7Ds3xoPj781wOK1CAtf0aLKfwOr5uz/g39JlylYrXmoOa55Uuh5+xGco26fJ3z5WX/+wz5d69ebvv1Ygdo1Si6VCy7icy9ry/z8DTouFKasQkAJmrpKwVqPNcpVDGtqHSRYVVmtqnJPrBkCSfXN5TUaMKYOnx82weRWQFRREEsTv6aFKFKvozNeB5BQpVm4/Ux3AOf9YTCbz6g37o0Hs+7cn0+78CMYDPrzSW8wZ1N/5k2DkW2Pp9PhLPD83nRCveE0GIz6zPe8MZuMYGQ4mI26bNjr0rzDiI3AZ3irtgZrnmMTcNwak71xa4ItQIK9MpmJQckYBW9p6pMVBYRgQUjEwYyJrGnvkB1sAqmKKwi443juTcJ93cGSw+I+9qASy5KIw+fDofzyBjsH5ARDKGAEiE6aUN5Rex/KEg/Hkbl+fwSc7fX7ikGyWs+BeArYA1APHuWMYaNG5TcWRW7cG7voSkxW2KpPQRRy6eSQqeeoliJh1FtCXonbAUfU89NPH0nRYsT4LiEMVm4EIA1L/yxKUjRMGfXv28EakxTNMkBdmIdOj8/2/jqVIrPJyQrdjKzhUWhipsPOG3unjaTqyh+juVL32JDj5slGh9iy3OydYitoXzUGQaMA9+Sx0uRWYzcyp961Qet2yVXv8xbqJISFwC6hC6ApIGcCGoCqTthqPlARrqTq54xLtc0GrV4P9DYbtvqzlxQXuCA/OCLS+nI9xBJ7DkW3mgeMPYIxyOYVE0l4w+p6Qzq5gzLcL1N3Ea6rTSoW3/A0iSNQkstiOg9Ldveq9WgC4THlPiwH5YGuvsdZr/POFv7IM7y5GdndraFhr+PblK6wUqlhQLc/CIeoui0TVMRfPpxLVbFoznwfFPrp/O9SqVoQBOFOpTAAkwO8HrmT0Z7spRekMtmcbaOPYVtiDXgcFIx1HZDNkoRYV+9/wWa9+/HEPT4+POhdkWQFSB8gfQvNpiSF1lCcFXF+bhrIKu7dJf5qSMn7o+PjpjYT1D4oEPezZBuc7L6FD7PjU0gAe+KqzWNiqTMWBo4jC1HZeVA2kEtaEm2Rj1AENbdKqQ75BZZJ9rTBizUAN1mxEEpimqIvFEdS1VZOuSi1SmoQQGxyhXtdoRDn9xlrw6w2fsmLqtzTDb2DhkLu3evK508XfuJZSw56jZuXOm2XUqo9ZCkNQ4ZGqtv2i6fNWebBF8VfwZ/abg7k7kVPAIYssEtErJXZAsGq1Av2AxIbLCOi/0jSFqmOccD1zY21UUQV0gUir4HKpEVGTfSxssF3Q0EwwmpUrLbRtLlwIaArII6RaDCdtqZkb9idYECCkXp3kyaj4fNOBXqhno05+ypu55xui5wclDPqwsSTR3h39NPRqQs5yD09Ovvw7vPhL+o4ArthJtZ7at+X42MdTxug8g+yt0HlGUusFqcVtRvJpNLM3eDk8PwcmDj59awqn00LGrplVj7AgXLys3qQ5MGYhi+RT9nii7mHtLdhb4aAZTjsKcDytLmZkKVVN4wXKrWzC4DTqprattRfVIsJJ1QncHtOc4MNQ02bDH6hE+yucrxKqDyp6kOBhN6kN0RZ9SaDcas3fVpaEuAsGYTW1EBrumBS0EyQiN5rVAfzeFoFc9T3JaAzEmKO7OqhWyVt1VizzuHWroC8tAUNVO87DOztlTJuWjn+dZxTFtI7CQa2staZuoTOQStkqOyWMcjvgKuXkFT4P8FPirRFsBchKyphFzRKYp/KnCTvTlP9I0YMIe/FhYSINjkHyRRvAehbfgIVkiC0JCeAecgF6pq/I3vCHX29LxGKvCCWkBOhASBlkgR5s7G9CJM5DUtiZpKM1pk8xMtp8sX7QqmbFqngvabpgU+oqOoztsgANFly7mYs16TLeqbUpw4YCF0Lmysk258A+PBZewEBUcM3WUbotxT8lN9gtUESz1uvaOzdky9rQHq2wtaTwQRLtv5k0m31Rjm4RnycrJibIUsCm1f6HrxFliB/1+dRMRCk7Is7pwJgQTDoS/O1/sa87+HH6xb5G+Q70Kis9EQuL1nVraEAdJzvsDjT2sE5Wi3AQH5Nlr+wsR39XYio+SsVLhiKCyIFbbsLbNtXu2fghizNXlmvnklrlfaiRH9moswpvEhA/f/j6zUD8t4KR5q1ZF59Ix/G9dfvplPpz9bwIysO0x6xpOARXUBZsBaIK7wQ355hd9TLQmWORn0hyuBcktHsCPKff/0bIom8jlRXXUXIQjL2KrtT70hhgJCxQcFHo/LUdwhYYcrKgrz/9Nl+shET8vlWA0aN6cbLbBKw2XxKg0kwGw4GY9brD4ds3u/ORvOuPx1NhuP+lHkj26Zd6gfj6ag7HA284aw/m3Q9Nh30J5T6PTrqD/2ABXM6eLLxovfdarjocXTcUQ/ddoTvW6HTSo+SLTXwqYcz9Q2Upr6oy1TsjxiV88Y9ayG3n+SLNqcf38sbV1my4FUQRnw+D/U2IND9vGw5IBmPWBtnM99IdurCVpcK6/EwrxRwkzcvFEiYo/4sipe1qgkYX9iqS0uaH3ltld7o1sQT9VN+japSFDCObyPhC14b7Z0ya8ksdIVucIU3XPMEC740EvJay0vYHRfyFgDvznRyKqnlxqgyElhjNX1VWAMIyWXcfSFtPX/vXS3RVO5ATtQ4SpGlbdlnlQkb2zYgL+xDYREPkGC9WJKLq+o9PaToq0uVOWYzNMBet4K2zNlbyVG3C/G1AvlSQZnUSpQibwdVSylvPBkWITVfD11KwFJS09eShXJLnZZAp3zz8H+DEKpvodQ1I7AEyO89D6SkNjCDfrgxWkixOizvsl2sPa2vX/P9HEepBO+4UUEAfNFUecZpiNJr5Df9lauiXF8vnjzvQRlr88b3Vpo4km9NkCsEd1ck01fr8rUK2UJBhelOkgSCEATkDGAT4hlAb3CeKvB+c3Kmptjk8IbyEHkhNAB7kOabm2752sHVpXo9M8VWamRkDHUpiIlK7whOv4JoyWjUInPw3gCsLWuzGJjFtJO/HgvUaEaWSegrTDWUVVt/PG31uoZraLfAELfpJ7p1GUulWLv4ckVRC9SbT6WClDo9IG/lO6Gy4SlfuWjXm1iRdZVQrF1cXtH8s/Nf6Cbhn6S42QvAEKBe8bGfsyazpN9sA3HgZrfo+BgPMa7FTL17u16RUL42G/Lr/PVXsKILwSP/Uj5ySEwj1V0D9/dvAaNUSMkogy8xQGUSyjqCi7Z6cQHTCGCNGy4wF9o7/wVE19sIGi4AAA=="""
MODEL_URL = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q8_0.gguf?download=true"
MODEL_BYTES = 675710816
MODEL_SHA256 = "ca59ca7f13d0e15a8cfa77bd17e65d24f6844b554a7b6c12e07a5f89ff76844e"
ROOT = Path("/kaggle/working/wave119")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
TARGET = ROOT / "target"
MODEL = ROOT / "qwen2.5-0.5b-instruct-q8_0.gguf"
FINAL_ZIP = Path("/kaggle/working/glcuda-t4-wave119-in-process-stability-results.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)
RESULTS.mkdir(parents=True)

def run(cmd, *, cwd=None, env=None, timeout=14400, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({k: str(v) for k, v in env.items()})
    p = subprocess.run([str(x) for x in cmd], cwd=cwd, env=merged, text=True,
                       capture_output=True, timeout=timeout)
    print("$", " ".join(str(x) for x in cmd), flush=True)
    if p.stdout:
        print(p.stdout[-12000:], flush=True)
    if p.stderr:
        print(p.stderr[-12000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p

def save(name, p):
    (RESULTS / name).write_text(
        f"RETURN_CODE {p.returncode}\n\nSTDOUT\n{p.stdout}\n\nSTDERR\n{p.stderr}",
        encoding="utf-8",
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    digest = sha256_file(FINAL_ZIP)
    print("ARCHIVE", FINAL_ZIP, digest, flush=True)
    return digest

def percentile(values, q):
    values = sorted(values)
    x = (len(values) - 1) * q
    lo, hi = math.floor(x), math.ceil(x)
    return values[lo] if lo == hi else values[lo] * (hi - x) + values[hi] * (x - lo)

def bootstrap_ci(values, seed=118, draws=20000):
    rng = random.Random(seed)
    n = len(values)
    medians = [statistics.median(values[rng.randrange(n)] for _ in range(n))
               for _ in range(draws)]
    return [percentile(medians, 0.025), percentile(medians, 0.975)]

phase = "bootstrap"
try:
    embedded = gzip.decompress(base64.b64decode(PATCH_GZIP_B64))
    if hashlib.sha256(embedded).hexdigest() != PATCH_SHA256:
        raise RuntimeError("embedded patch hash mismatch")
    patch_path = RESULTS / "wave119.patch"
    patch_path.write_bytes(embedded)
    (RESULTS / "source.json").write_text(json.dumps({
        "build": BUILD, "base_rev": BASE_REV, "source_rev": SOURCE_REV,
        "patch_sha256": PATCH_SHA256, "patch_bytes": len(embedded),
    }, indent=2), encoding="utf-8")

    gpu = run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
               "--format=csv,noheader,nounits"], timeout=60)
    save("nvidia-smi.log", gpu)
    fields = [x.strip() for x in gpu.stdout.splitlines()[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"requires Tesla T4 sm_75, got {fields}")

    phase = "reconstruct"
    clone = run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800)
    save("git-clone.log", clone)
    checkout = run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=600)
    save("git-checkout.log", checkout)
    applied = run(["git", "apply", "--whitespace=error", patch_path], cwd=TREE)
    save("git-apply.log", applied)
    diff = run(["git", "diff", "--check"], cwd=TREE)
    save("git-diff-check.log", diff)

    cargo_candidates = [shutil.which("cargo"), Path.home() / ".cargo/bin/cargo",
                        "/usr/local/cargo/bin/cargo", "/opt/conda/bin/cargo"]
    cargo = next((str(x) for x in cargo_candidates if x and Path(x).is_file()), None)
    cargo_env = {}
    bootstrapped = False
    if cargo is None:
        bootstrapped = True
        rustup_script = ROOT / "rustup-init.sh"
        urllib.request.urlretrieve("https://sh.rustup.rs", rustup_script)
        cargo_home = ROOT / "cargo-home"
        rustup_home = ROOT / "rustup-home"
        cargo_env = {"CARGO_HOME": cargo_home, "RUSTUP_HOME": rustup_home}
        install = run(["bash", rustup_script, "-y", "--profile", "minimal",
                       "--default-toolchain", "stable", "--no-modify-path"],
                      env=cargo_env, timeout=1800)
        save("rustup-install.log", install)
        cargo = str(cargo_home / "bin/cargo")
    if not Path(cargo).is_file():
        raise RuntimeError(f"cargo unavailable after bootstrap: {cargo}")
    (RESULTS / "cargo-discovery.json").write_text(json.dumps({
        "selected": cargo, "bootstrapped": bootstrapped,
        "candidates": [str(x) for x in cargo_candidates if x],
    }, indent=2), encoding="utf-8")
    common = {**cargo_env, "CARGO_TARGET_DIR": TARGET, "CUDA_VISIBLE_DEVICES": "0"}

    phase = "host-tests"
    tests = run([cargo, "test", "-p", "glcuda", "--lib", "--locked"], cwd=TREE, env=common)
    save("cargo-lib-tests.log", tests)
    if "67 passed" not in tests.stdout or "0 failed" not in tests.stdout:
        raise RuntimeError("unexpected host test summary")

    phase = "cuda-parity"
    parity = run([cargo, "test", "--release", "-p", "glcuda", "--test", "parity",
                  "--locked", "--", "--nocapture", "--test-threads=1"],
                 cwd=TREE, env=common, check=False)
    save("cargo-cuda-parity.log", parity)
    if parity.returncode or "0 failed" not in parity.stdout:
        raise RuntimeError("CUDA parity failed")

    phase = "compiler-resource"
    ptxas = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    resource = run([ptxas, "-v", "-arch=sm_75", TREE / "glcuda/src/kernels/glcuda_sm75.ptx",
                    "-o", ROOT / "wave119.cubin"], check=False)
    save("ptxas-sm75.log", resource)
    if resource.returncode or "spill stores" not in resource.stderr:
        raise RuntimeError("ptxas resource gate failed")

    phase = "model"
    urllib.request.urlretrieve(MODEL_URL, MODEL)
    model_meta = {"bytes": MODEL.stat().st_size, "sha256": sha256_file(MODEL)}
    if model_meta != {"bytes": MODEL_BYTES, "sha256": MODEL_SHA256}:
        raise RuntimeError(f"model identity mismatch: {model_meta}")
    (RESULTS / "model.json").write_text(json.dumps(model_meta, indent=2), encoding="utf-8")

    phase = "build"
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave118_in_process_stability", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)
    exe = TARGET / "release/examples/wave118_in_process_stability"
    prod_env = {**common, "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
                "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_NTILE128": "1",
                "GLCUDA_BSTAGE": "1", "GLCUDA_GEMM_N16": "1",
                "GLCUDA_GEMM_N16_PREFETCH": "1", "GLCUDA_ATTN_MMA4": "1",
                "GLCUDA_ATTN_MMA4_REGQ": "1", "GLCUDA_ATTN_MMA4_AV": "1"}

    phase = "build"
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave118_in_process_stability", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)
    exe = TARGET / "release/examples/wave118_in_process_stability"
    prod_env = {**common, "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
                "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_NTILE128": "1",
                "GLCUDA_BSTAGE": "1", "GLCUDA_GEMM_N16": "1",
                "GLCUDA_GEMM_N16_PREFETCH": "1", "GLCUDA_ATTN_MMA4": "1",
                "GLCUDA_ATTN_MMA4_REGQ": "1", "GLCUDA_ATTN_MMA4_AV": "1",
                "GLCUDA_TELEMETRY": "1"}

    phase = "event-profile"
    measured = run([exe, MODEL, "profile"], cwd=TREE, env=prod_env, check=False)
    save("event-profile.log", measured)
    if measured.returncode or "[wave119-profile]" not in measured.stdout:
        raise RuntimeError("single-pass production profile failed")

    phase = "sass"
    nvdisasm = shutil.which("nvdisasm") or "/usr/local/cuda/bin/nvdisasm"
    sass = run([nvdisasm, ROOT / "wave119.cubin"], check=False)
    save("nvdisasm.log", sass)

    phase = "ncu"
    ncu_candidates = [shutil.which("ncu"), "/usr/local/cuda/bin/ncu",
                      "/opt/nvidia/nsight-compute/ncu"]
    ncu = next((str(x) for x in ncu_candidates if x and Path(x).is_file()), None)
    counter_status = "unavailable"
    profiles = {}
    metrics = ",".join([
        "gpu__time_duration.sum", "sm__warps_active.avg.pct_of_peak_sustained_active",
        "smsp__warps_eligible.avg.per_cycle_active", "smsp__inst_executed.avg.per_cycle_active",
        "sm__pipe_tensor_cycles_active.avg.pct_of_peak_sustained_active",
        "l1tex__data_pipe_lsu_wavefronts_mem_shared_op_ld.sum",
        "dram__bytes_read.sum", "lts__t_bytes_srcunit_tex_op_read.sum",
        "smsp__warp_issue_stalled_long_scoreboard_per_warp_active.pct",
        "smsp__warp_issue_stalled_barrier_per_warp_active.pct",
        "smsp__warp_issue_stalled_wait_per_warp_active.pct",
    ])
    targets = {
        "ffn": "regex:gl_gemm_mma_q8_bstage_n16.*",
        "attention": "regex:gl_attn_mma4_regq_avmma_fused_f32",
    }
    if ncu:
        query = run([ncu, "--query-metrics", "--chip", "tu104"], check=False, timeout=600)
        save("ncu-query.log", query)
        for name, kernel in targets.items():
            prof = run([ncu, "--csv", "--page", "raw", "--target-processes", "all",
                        "--replay-mode", "application", "--cache-control", "none",
                        "--kernel-name-base", "function", "--kernel-name", kernel,
                        "--launch-count", "1", "--metrics", metrics,
                        exe, MODEL, "profile"], cwd=TREE, env=prod_env,
                       check=False, timeout=14400)
            save(f"ncu-{name}.log", prof)
            profiles[name] = prof.returncode
        joined = "\n".join((RESULTS / f"ncu-{x}.log").read_text(encoding="utf-8")
                           for x in targets)
        if "ERR_NVGPUCTRPERM" in joined or "permission" in joined.lower():
            counter_status = "permission-denied"
        elif profiles and all(value == 0 for value in profiles.values()):
            counter_status = "collected"
        else:
            counter_status = "failed"

    summary = {"wave": 119, "gpu": fields, "model": model_meta,
               "event_profile_ok": True, "ncu": ncu,
               "counter_status": counter_status, "profile_returncodes": profiles,
               "sass_available": sass.returncode == 0,
               "retention_authority": False, "target_15000_tps_achieved": False}
    (RESULTS / "wave119-summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("WAVE119_RESULT", json.dumps(summary, indent=2), flush=True)
    archive()
except Exception:
    (RESULTS / "FAILED.json").write_text(json.dumps({
        "phase": phase, "traceback": traceback.format_exc()}, indent=2), encoding="utf-8")
    archive()
    raise
